#### N7 — FinBERT Text Embeddings (M2)

Embeds Gemini 2.5 Flash business summaries using FinBERT (ProsusAI/finbert)
→ 768-dimensional mean-pooled, L2-normalised embeddings per firm-year.

**Input:** `business_summaries_{year}.csv` (N3)
**Output:** `text_embeddings.parquet` — schema: `tic | fyear | emb_0 … emb_767`
**Runtime:** ~45 min GPU / ~3–4h CPU


In [ ]:
# imports & config
import sys
from pathlib import Path

notebook_dir = Path('__file__').parent if '__file__' in dir() else Path.cwd()
repo_root = next(
    (p for p in [notebook_dir, *notebook_dir.parents] if (p / 'config.py').exists()),
    None
)
if repo_root is None:
    raise FileNotFoundError('config.py not found — check repo structure')
sys.path.insert(0, str(repo_root))

from config import *
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Config loaded.')
print(f'  SUMMARIES_DIR : {SUMMARIES_DIR}')
print(f'  EMBEDDINGS    : {EMBEDDINGS}')
print(f'  MODEL         : {EMBEDDING_MODEL}')
print(f'  Device        : {device}')


In [ ]:
# declare I/O
INPUTS  = list(SUMMARIES_FILES.values())
OUTPUTS = [EMBEDDINGS]
# PURPOSE : FinBERT embeddings for Gemini summaries — 768-dim per firm-year
# RUNTIME : ~45 min GPU / ~3-4h CPU
# DEPENDS : business_summaries_{year}.csv (N3)


### 1 · Load & Validate Summaries


In [ ]:
# load all summaries and validate
INVALID_FLAGS = {'ERROR', 'INSUFFICIENT_DATA', 'ERROR_EXTRACTING_TEXT'}

frames = []
for yr in YEARS:
    path = SUMMARIES_FILES[yr]
    if not path.exists():
        print(f"  {yr}: FILE NOT FOUND — {path}")
        continue
    df_yr = pd.read_csv(path, usecols=['tic', 'fyear', SUMMARY_COL])
    df_yr = df_yr[
        df_yr[SUMMARY_COL].notna() &
        ~df_yr[SUMMARY_COL].isin(INVALID_FLAGS) &
        (df_yr[SUMMARY_COL].str.len() > 50)
    ].copy()
    frames.append(df_yr)
    print(f"  {yr}: {len(df_yr):,} valid summaries")

df_text = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(df_text):,} firm-year summaries")
print(f"Unique firms: {df_text['tic'].nunique():,}")
print(f"Years: {sorted(df_text['fyear'].unique())}")


### 2 · Load FinBERT


In [ ]:
# load FinBERT tokenizer and model
print(f"Loading {EMBEDDING_MODEL}...")
tokenizer = BertTokenizer.from_pretrained(EMBEDDING_MODEL)
model     = BertModel.from_pretrained(EMBEDDING_MODEL, ignore_mismatched_sizes=True).to(device)
model.eval()
print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")

### 3 · Embed — Year by Year


In [ ]:
BATCH_SIZE = 64

def embed_texts(texts, tokenizer, model, device, batch_size=BATCH_SIZE):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch,
            max_length=512,
            truncation=True,
            padding=True,
            return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            output = model(**encoded)
        hidden = output.last_hidden_state                        # (batch, seq, 768)
        mask   = encoded['attention_mask'].unsqueeze(-1).float()
        emb    = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)  # mean pool
        emb    = emb.cpu().numpy()
        norms  = np.linalg.norm(emb, axis=1, keepdims=True)
        emb    = emb / np.where(norms == 0, 1, norms)
        all_embeddings.append(emb)
    return np.vstack(all_embeddings)

print(f'embed_texts() defined.')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  Max length : 512 tokens (truncation=True)')
print(f'  Pooling    : mean (non-padding tokens)')
print(f'  Normalise  : L2 per embedding')


In [ ]:
yearly_parquets = []

for yr in YEARS:
    checkpoint = DATA_PROC / f'text_embeddings_{yr}.parquet'

    if checkpoint.exists():
        print(f'  {yr}: checkpoint found — loading')
        yearly_parquets.append(pd.read_parquet(checkpoint))
        continue

    df_yr = df_text[df_text['fyear'] == yr].reset_index(drop=True)
    if len(df_yr) == 0:
        print(f'  {yr}: no summaries — skipping')
        continue

    print(f'  {yr}: embedding {len(df_yr):,} summaries...')
    texts = df_yr[SUMMARY_COL].tolist()
    embs  = embed_texts(texts, tokenizer, model, device)

    # Build parquet: tic | fyear | emb_0 ... emb_767
    emb_cols = {f'emb_{i}': embs[:, i] for i in range(EMBEDDING_DIM)}
    df_emb = pd.DataFrame({'tic': df_yr['tic'].values,
                           'fyear': df_yr['fyear'].values,
                           **emb_cols})
    df_emb.to_parquet(checkpoint, index=False)
    yearly_parquets.append(df_emb)
    print(f'  {yr}: done → {checkpoint.name}')

# Concat all years
df_embeddings = pd.concat(yearly_parquets, ignore_index=True)
df_embeddings.to_parquet(EMBEDDINGS, index=False)

print(f'\nSaved: {EMBEDDINGS}')
print(f'Shape: {df_embeddings.shape[0]:,} rows × {df_embeddings.shape[1]} columns')
print(f'  tic + fyear + {EMBEDDING_DIM} embedding dimensions')


In [ ]:
# validate embeddings
emb_cols = [f'emb_{i}' for i in range(EMBEDDING_DIM)]

# Check L2 norms ≈ 1.0
norms = np.linalg.norm(df_embeddings[emb_cols].values, axis=1)
print(f"L2 norm check (should be ~1.0):")
print(f"  Mean: {norms.mean():.6f}")
print(f"  Std : {norms.std():.6f}")
print(f"  Min : {norms.min():.6f}")
print(f"  Max : {norms.max():.6f}")

print(f"\nNaN check: {df_embeddings[emb_cols].isna().sum().sum()} NaN values")
print(f"\nFirm-years per year:")
print(df_embeddings.groupby('fyear')['tic'].count().to_string())


In [ ]:
# final summary
print("=" * 60)
print("N7 COMPLETE — FINBERT EMBEDDINGS SUMMARY")
print("=" * 60)
print(f"  Model     : {EMBEDDING_MODEL}")
print(f"  Dimension : {EMBEDDING_DIM}")
print(f"  mean (non-padding tokens)")
print(f"  Normalised: L2 per embedding")
print(f"  Firm-years: {len(df_embeddings):,}")
print(f"  Output    : {EMBEDDINGS}")
print()
print("  Next: N8 — kNN text peer lists (M2)")
